# Simple Pendulum

**Author:** Faizur Rahman  
**Institute:** BIT Sindri  
**Department:** Chemical Engineering  
**Registration No.:** 23030420034  
**Email:** faizurr464@gmail.com  
**Instructor:** Prof. Ch V Raghunath

## 1. Introduction

A simple pendulum is an idealised model: a point mass $m$ suspended by an unstretchable, massless string (or rod) of length $L$. Let $\theta$ denote the angle between the vertical axis and the string, so $\theta = 0$ is equilibrium.

Two forces act on the mass: gravity $F = mg$ and the string pull $S$. Decomposed into radial ($\hat{r}$) and azimuthal ($\hat{\theta}$) directions, only the azimuthal component drives motion:

$$F_\theta = -mg \sin\theta. \tag{1}$$

The radial gravity component is cancelled by tension: $F_r = -S$. With tangential acceleration $a = L\ddot{\theta}$, Newton's second law gives:

$$L\ddot{\theta} = -g\sin\theta \quad\Rightarrow\quad \ddot{\theta} + \frac{g}{L}\sin\theta = 0. \tag{3}$$

## 2. Analytical Approximation

Equation (3) is nonlinear and has no elementary closed-form solution. For small angles $\theta \ll 1$, $\sin\theta \approx \theta$, so:

$$L\ddot{\theta} \approx -g\theta, \tag{4}$$

with solution

$$\theta(t) = \theta_0 \cos(\omega t), \qquad \omega = \sqrt{\frac{g}{L}}, \qquad T = 2\pi\sqrt{\frac{L}{g}}. \tag{5}$$

This is simple harmonic motion.

## 3. Numerical Solution

Introduce angular velocity $\omega = \dot{\theta}$ and rewrite as two first-order ODEs:

$$d\omega = -\frac{g}{L}\sin\theta\, dt, \qquad d\theta = \omega\, dt. \tag{6--7}$$

With a small timestep $\Delta t$, use the forward Euler method from $\theta(0)=\theta_0$ to advance $n$ steps to time $t = n\Delta t$.

## 4. Sample Problem

A $40\,\mathrm{kg}$ steel ball (radius $r=10\,\mathrm{cm}$) suspended by a $25\,\mathrm{m}$ steel wire. Small-amplitude period:

$$T = 2\pi\sqrt{\frac{L}{g}} \approx 10\,\mathrm{s}.$$

## 5. Python Implementation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.style.use("bmh")
figsize = (12, 5)
dpi = 150

g = 9.81  # m/s^2
L = 25    # m
m = 40    # kg

print(f"Small-amplitude period T = {2 * np.pi * np.sqrt(L / g):.3f} s")

: 

In [ ]:
def analytical_approximation(t, theta0):
    """Evaluates the analytical (small-angle) approximation."""
    return theta0 * np.cos(t * (g / L) ** 0.5)


def RHS(theta, w, dt):
    """Return the right-hand side of the ODE describing a simple pendulum."""
    dw = -np.sin(theta) * dt * g / L
    dtheta = w * dt
    return dw, dtheta


def integrate_one_step(theta, w, dt):
    """Performs one step of integration (forward Euler)."""
    dw, dtheta = RHS(theta, w, dt)
    w = w + dw
    theta = theta + dtheta
    return w, theta


def integrate_n_steps(theta0, w0, dt, n):
    """Performs integration for n time steps."""
    theta = [0.0] * (n + 1)
    w = [0.0] * (n + 1)
    theta[0] = theta0
    w[0] = w0
    for i in range(n):
        w[i + 1], theta[i + 1] = integrate_one_step(theta[i], w[i], dt)
    return w, theta

Compare numerical solutions for $\theta_0 = 15^\circ$ and $60^\circ$ with the analytical approximation.

In [ ]:
theta0_1 = np.pi / 12  # 15°
theta0_2 = np.pi / 3   # 60°

T = 20
n = 10000
t = np.linspace(0, T, n + 1)
dt = T / float(n)

w1, theta1 = integrate_n_steps(theta0_1, 0, dt, n)
w2, theta2 = integrate_n_steps(theta0_2, 0, dt, n)

plt.figure(figsize=figsize, dpi=dpi)
plt.title("Angular position")
plt.plot(t, theta1, "m", label=r"$\theta_0 = %.0f^\circ$" % (theta0_1 * 180 / np.pi))
plt.plot(t, analytical_approximation(t, theta0_1), "m--", label=r"Approximation")
plt.plot(t, theta2, "g", label=r"$\theta_0 = %.0f^\circ$" % (theta0_2 * 180 / np.pi))
plt.plot(t, analytical_approximation(t, theta0_2), "g--", label=r"Approximation")
plt.xlabel(r"$t$, [s]")
plt.ylabel(r"$\theta(t)$, [rad]")
plt.legend()
plt.show()

The approximation for the small initial angle is quite good, but as the initial angle increases the approximation becomes less accurate.

## 6. Conservation of Energy

Total mechanical energy

$$E = U + K = mgL(1 - \cos\theta) + \tfrac{1}{2} m L^2 \dot{\theta}^2 \tag{8}$$

should be conserved. Plotting kinetic, potential, and total energy checks whether $\Delta t$ was small enough.

In [ ]:
def compute_PE(theta):
    return m * g * L * (1 - np.cos(theta))


def compute_KE(w):
    return 0.5 * m * L**2 * np.asarray(w) ** 2


plt.figure(figsize=figsize, dpi=dpi)
plt.title(r"Mechanical energy, $\theta_0 = %.0f^\circ$" % (theta0_2 * 180 / np.pi))
plt.plot(t, compute_PE(theta2), label=r"Potential energy")
plt.plot(t, compute_KE(w2), label=r"Kinetic energy")
plt.plot(t, compute_PE(theta2) + compute_KE(w2), label=r"Total energy")
plt.xlabel(r"$t$, [s]")
plt.ylabel(r"$E$, [J]")
plt.legend(loc=1)
plt.show()

In [ ]:
def compute_error(w, theta):
    """Computes the relative error in total energy."""
    E0 = compute_PE(theta[0]) + compute_KE(w[0])
    E1 = compute_PE(theta[-1]) + compute_KE(w[-1])
    return np.abs((E0 - E1) / E0)


print("Relative change in E:")
print("Theta = %.0f: %.2e" % (theta0_1 * 180 / np.pi, compute_error(w1, theta1)))
print("Theta = %.0f: %.2e" % (theta0_2 * 180 / np.pi, compute_error(w2, theta2)))